In [ ]:
import geopandas as gpd

print("Loading NYC road network...")
roads = gpd.read_file("../data/external/nyc_roads.geojson")

print(f"Segments: {len(roads):,}")
print(f"CRS: {roads.crs}")
print(f"Columns: {list(roads.columns)}")

roads.to_file("../data/external/nyc_roads.gpkg", driver="GPKG")
print("Saved as nyc_roads.gpkg")

ModuleNotFoundError: No module named 'pyrosm'

In [2]:
pip install pyrosm

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
     ---- ----------------------------------- 0.3/2.5 MB ? eta -:--:--
     --------------------- ------------------ 1.3/2.5 MB 3.9 MB/s eta 0:00:01
     ----------------------------- ---------- 1.8/2.5 MB 3.5 MB/s eta 0:00:01
     -------------------------------------- - 2.4/2.5 MB 3.2 MB/s eta 0:00:01
     ---------------------------------------- 2.5/2.5 MB 3.0 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: still running...
  Installing build dependencies: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × installing build dependencies for pyrosm did not run successfully.
  │ exit code: 1
  ╰─> [71 lines of output]
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'done'
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Installing backend dependencies: started
        Installing backend dependencies: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status '

In [4]:
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import shape
from pathlib import Path

print("Downloading NYC road centerlines in chunks...")

CHUNK = 10000
MAX = 130000
BASE_URL = "https://data.cityofnewyork.us/resource/inkn-q76z.geojson"

all_features = []
offset = 0

while offset < MAX:
    params = {"$limit": CHUNK, "$offset": offset}
    r = requests.get(BASE_URL, params=params, timeout=60)
    data = r.json()
    features = data.get("features", [])
    if not features:
        break
    all_features.extend(features)
    offset += CHUNK
    print(f"  {offset:,} rows done")

print(f"Total features: {len(all_features):,}")

# convert to GeoDataFrame
from shapely.geometry import shape
import geopandas as gpd

geometries = [shape(f["geometry"]) for f in all_features]
properties = [f["properties"] for f in all_features]

roads = gpd.GeoDataFrame(properties, geometry=geometries, crs="EPSG:4326")
print(f"Columns: {list(roads.columns)}")

roads.to_file("../data/external/nyc_roads.gpkg", driver="GPKG")
print("Saved as nyc_roads.gpkg")

  10,000 rows done
  20,000 rows done
  30,000 rows done
  40,000 rows done
  50,000 rows done
  60,000 rows done
  70,000 rows done
  80,000 rows done
  90,000 rows done
  100,000 rows done
  110,000 rows done
  120,000 rows done
  130,000 rows done
Total features: 122,272
Columns: ['stname_label', 'rw_type', 'segment_type_value', 'l_low_hn', 'sandist_ind', 'bphys_id', 'pre_directional', 'rsubsect', 'number_park_lanes', 'avgtravtime', 'borough_indicator', 'created_date', 'r_zip', 'r_low_hn', 'bike_lane', 'nominaldir', 'post_type', 'rwjurisdiction', 'joinid', 'nonped', 'l_blockfaceid', 'pre_type', 'post_directional', 'collectionmethod', 'fire_lane', 'objectid', 'globalid', 'segment_type', 'street_name', 'within_bndy_dist', 'truck_route_type', 'b5sc', 'full_street_name', 'shape_length', 'streetwidth_irr', 'status', 'posted_speed', 'carto_display_level', 'r_high_hn', 'l_zip', 'segmentlength', 'streetwidth', 'lsubsect', 'special_disaster', 'r_blockfaceid', 'fcc', 'boroughcode', 'to_level_